In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [ ]:
df  = pd.DataFrame()

## Sentiment

In [ ]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)

df_sentiment = df_sentiment.ffill()

df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm10"]=df_sentiment["sentiment"].ewm(10).mean()
df_sentiment["sentiment_ewm20"]=df_sentiment["sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

# PRESS RELEASE COUNT
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount(t-1)"].ffill()
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount(t-2)"].ffill()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount(t-1)"]/df_sentiment["PressReleaseCount(t-2)"])


In [ ]:
df_sentiment.loc[df_sentiment["compound"]<0.9]

In [ ]:
# df_sentiment = df_sentiment[["Date", "sentiment", "sentiment(t-1)", "sentiment(t-2)", "sentiment_return", "sentiment_ln_return", "sentiment_ewm", "PressReleaseCount"]]

In [ ]:
df = df_sentiment

In [ ]:
df.sort_index()

## VIX

In [ ]:
# from yahoo_fin.stock_info import get_data

# df_vix = get_data("^VIX", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d")
# df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
# df_vix.drop(columns=["high", "low", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return_c2c"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
df_vix["vix_return_c2c"]=(df_vix["close"]-df_vix["open"])/df_vix["open"]
# vix LN RETURN
df_vix["vix_ln_return_c2c"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"]=np.log(df_vix["close"]/df_vix["open"])

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [ ]:
df_vix.tail(50)

In [ ]:
aa = df_vix.head(10)

In [ ]:
df = pd.merge(df_vix,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [ ]:
df

## S&P500

In [ ]:
# from yahoo_fin.stock_info import get_data

# df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d")
# df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)

df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [ ]:
df = pd.merge(df_sp500,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [ ]:
df

In [ ]:
# df = df.drop([475])

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# TEXTBLOB

In [ ]:
# Reading the saved data pickle file

# df_polarity = pd.DataFrame()

# df_polarity=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentArticlesTextBlob.pkl', compression=None)
# df_polarity.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_polarity.drop(columns=["Company", "PressReleaseCount", "Articles", "Title", "Text"], inplace=True)

# df_polarity.index = pd.to_datetime(df_polarity.index)
# df_polarity.Date = pd.to_datetime(df_polarity.Date)

# df_polarity = df_polarity.ffill()

# df_polarity["subjectivity(t-1)"] = df_polarity["subjectivity"].shift(periods=1)
# df_polarity["subjectivity(t-2)"] = df_polarity["subjectivity"].shift(periods=2)

# # polarity RETURN
# df_polarity["subjectivity_return"]=(df_polarity["subjectivity(t-1)"]-df_polarity["subjectivity(t-2)"])/df_polarity["subjectivity(t-2)"]
# # polarity LN RETURN
# df_polarity["subjectivity_ln_return"]=np.log(df_polarity["subjectivity(t-1)"]/df_polarity["subjectivity(t-2)"])
# # polarity EWM
# df_polarity["subjectivity_ewm"]=df_polarity["subjectivity"].ewm(10).mean()

# df.reset_index(drop=True, inplace=True)
# df_polarity.reset_index(drop=True, inplace=True)
# df_polarity = df_polarity.ffill()

# df = pd.merge(df_polarity,df,how='outer', on='Date')
# df = df.ffill()
# df = df.sort_values(by="Date")

In [ ]:
df.index = df.Date

In [ ]:
df

# THREE PERIODS

In [ ]:
df = df.ffill()
df.index = df.Date

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-09-28']
df_part3 = df_part3.ffill()

In [ ]:
df_part3

In [ ]:
df_part3.day